# 22 - Rerouting and Load Redistribution

Every earlier resilience notebook in this project answers the question *does the network fall apart?* - it removes a station and measures fragmentation (articulation points, largest-component share, efficiency drop). That is a topological answer. It is not what a passenger experiences. A passenger whose station is shut does not usually become unreachable; they take a **longer route**. The cost of a closure is therefore measured in **extra minutes of travel**, and only in the worst cases in **lost connectivity**.

This notebook builds that missing model. For each of the top-N structurally critical stations it:

1. finds a sample of origin-destination (OD) pairs whose fastest route currently passes **through** that station;
2. deletes the station from the travel-time graph;
3. recomputes the fastest route for each of those pairs and measures the **detour in seconds**, plus how many pairs become **unreachable**;
4. records **which other stations now appear on the new routes** - the stations that absorb the displaced trips.

The result is a ranking of stations by **passenger-time cost** rather than by topology, and a direct comparison between the two rankings. They disagree, and the disagreement is the finding.

**Research question:** *when a critical station is shut, how much extra travel time does the network actually impose, on whom, and which neighbouring stations take the load?*

This answers the future-direction slide item "מודל ריאלי של ניתוב מחדש סביב תחנה מושבתת" (a realistic rerouting model around a disabled station).

## Inputs

* `outputs/nb/18_travel_time_network/tables/edges_traveltime.csv` - one row per directed segment with `from_stop, to_stop, median_travel_seconds, p25_travel_seconds, p75_travel_seconds, trip_frequency, n_observations`. **This is the required input** and it is produced by notebook 18. Without it there is no notion of "minutes", only hops, and the whole point of this notebook disappears.
* `outputs/nb/04_centrality_analysis/tables/stop_metrics.csv` - used only to choose which stations to shut down (top `approx_betweenness`) and to supply names, coordinates and regions.

No raw GTFS file is read here: the 816 MB `stop_times.txt` was already digested into travel times by notebook 18.

## Notebooks that must run first

`01` -> `02` -> `04` (for the criticality ranking) and **`18`** (for the travel-time graph). Nothing else.

## Outputs (everything under `outputs/nb/22_rerouting_model/`)

* `tables/rerouting_results.csv` - one row per shut-down station: `removed_stop, stop_name, mean_detour_seconds, median_detour_seconds, pairs_disconnected, reachable_share, top_absorbing_stops` (plus diagnostic columns).
* `tables/absorbing_stations.csv` - long-format table: for each removed station, every station that gained traffic and how many rerouted OD pairs now pass through it.
* `tables/rank_comparison.csv` - topological rank vs passenger-time rank, per station.
* `tables/detour_samples.csv` - the per-OD-pair detours behind the aggregates, so the distributions can be re-checked.
* `rerouting_summary.json` - headline numbers, all constants used, and the rank-agreement statistics.
* `figures/detour_cost_per_station.png`, `figures/topological_vs_passenger_time_rank.png`, `figures/detour_distribution.png`, `figures/absorbing_stations_worst.png`.

The frozen, report-cited folders `outputs/tables`, `outputs/figures` and `outputs/rail` are never touched.

## 1. Environment bootstrap

The cell below makes the notebook runnable both on a local checkout and on Google Colab. It defines `_ensure(...)`, which pip-installs only the packages that are genuinely missing (so re-running the notebook is cheap), and `find_repo_root()`, which walks up from the current directory looking for the GTFS folder and, failing that, clones the repository into `/content`. It then sets `REPO`, `DATA` and `OUT` and creates the notebook output root. Every later cell relies on these three paths, so this must run first.

In [ ]:
# --- Environment bootstrap (safe to re-run, works locally and on Google Colab) ---
import os, sys, subprocess
from pathlib import Path

def _ensure(*pkgs):
    """Install only the packages that are actually missing."""
    import importlib.util
    alias = {"scikit-learn": "sklearn", "python-louvain": "community",
             "python-bidi": "bidi", "node2vec": "node2vec"}
    missing = [p for p in pkgs
               if importlib.util.find_spec(alias.get(p, p.replace("-", "_"))) is None]
    if missing:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

def find_repo_root():
    """Find the repo locally; on Colab, clone it."""
    here = Path(os.getcwd()).resolve()
    for cand in [here, *here.parents]:
        if (cand / "israel-public-transportation").is_dir():
            return cand
    target = Path("/content/israel-transit-network-resilience")
    if not target.exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/seanfourman/israel-transit-network-resilience.git",
                        str(target)], check=True)
    return target

REPO = find_repo_root()
os.chdir(REPO)
DATA = REPO / "israel-public-transportation"
OUT = REPO / "outputs" / "nb"
OUT.mkdir(parents=True, exist_ok=True)
print("Repo root:", REPO)

## 2. Libraries, stage folders and the cost constants

We import the scientific stack plus `scipy.sparse.csgraph`. **Why scipy and not `networkx` for the shortest paths?** This notebook runs thousands of Dijkstra queries over a ~30,000-node / ~52,000-edge graph. `networkx` is pure Python and needs roughly 0.3 s per single-source Dijkstra; `scipy.sparse.csgraph.dijkstra` is compiled and answers a *multi-source* query (hundreds of sources at once) in about a second. That difference is what makes this analysis possible at all on a laptop. The graph is still built and reported through `networkx`-compatible tables, so nothing about the model changes.

All the expensive knobs live in this one cell:

| Constant | Default | What it costs |
| --- | --- | --- |
| `TOP_N_STATIONS` | 30 | One multi-source Dijkstra over the whole graph **per station**. See the note below on why a full sweep is out of reach. |
| `N_SOURCES` | 250 | The baseline all-pairs-from-sample Dijkstra. Memory is `N_SOURCES x n_nodes` for distances (float64, ~61 MB) plus the same again as int32 predecessors (~30 MB). Raising this to 1000 would need ~360 MB. |
| `TARGETS_PER_SOURCE` | 80 | 250 x 80 = 20,000 candidate OD pairs. Costs nothing extra in Dijkstra time - the distances are already computed - only path reconstruction. |
| `MAX_PAIRS_PER_STATION` | 400 | Caps the rerouting work for a very central station. Each rerouted pair needs one predecessor walk. |
| `DISCONNECT_PENALTY_S` | 3600 | The seconds charged to an OD pair that becomes **unreachable**. This is a judgement call, not a measurement - see the limitations section. |

**Why not sweep all 30,463 stations?** Each removed station requires rebuilding the sparse matrix and running a fresh multi-source Dijkstra (~1-2 s here), *and* the through-traffic index would have to be built for every node rather than for 30. A full sweep is therefore on the order of 10-17 hours of compute plus a much larger OD sample to give every station enough through-traffic to be measurable. The top-N restriction is a cost decision, and it is also a defensible one: stations that carry almost no through traffic have almost no detour cost by construction, so the interesting mass is in the head of the distribution. It does mean this notebook **cannot** discover a low-betweenness station with a surprisingly high passenger-time cost - that is a known blind spot, stated again in the limitations.

In [ ]:
# --- Libraries and stage folders ------------------------------------------
_ensure('pandas', 'numpy', 'networkx', 'scipy', 'matplotlib', 'seaborn')

import json
import time
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import connected_components, dijkstra
from scipy.stats import spearmanr

sns.set_theme(style='whitegrid', font_scale=1.05)

STAGE = OUT / '22_rerouting_model'
TABLES = STAGE / 'tables'
FIGURES = STAGE / 'figures'
TABLES.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)

# --- Tunable constants: the entire cost of this notebook lives here -------
TOP_N_STATIONS        = 30      # stations shut down, one at a time
N_SOURCES             = 250     # origins in the OD sample
TARGETS_PER_SOURCE    = 80      # destinations drawn per origin -> 20,000 OD pairs
MAX_PAIRS_PER_STATION = 400     # cap on rerouted pairs evaluated per removed station
MIN_EDGE_SECONDS      = 1.0     # floor on an edge weight (see note in section 5)
DISCONNECT_PENALTY_S  = 3600.0  # seconds charged to an OD pair that becomes unreachable
TOP_ABSORBING         = 5       # absorbing stations named per removed station
RANDOM_SEED           = 42
FIG_DPI               = 150

rng = np.random.default_rng(RANDOM_SEED)

print('stage folder :', STAGE)
print('OD sample    :', N_SOURCES, 'origins x', TARGETS_PER_SOURCE, 'destinations =',
      N_SOURCES * TARGETS_PER_SOURCE, 'candidate pairs')

## 3. Hebrew label rendering

Stop names in the Israeli GTFS feed are Hebrew, and several figures below print them (the per-station detour chart, the rank-comparison scatter, the absorbing-stations chart). Matplotlib does not implement the Unicode bidirectional algorithm, so right-to-left text comes out reversed and unreadable. The cell below monkey-patches `matplotlib.text.Text.set_text` once so that any string containing Hebrew characters is converted to display order via `python-bidi` before it is drawn, and selects a font that actually has Hebrew glyphs. It is idempotent - re-running it will not stack patches (a double patch would reverse the text back again).

In [ ]:
# Stop names are Hebrew. Matplotlib does not apply the Unicode bidi algorithm, so
# Hebrew labels render reversed. Patch it once, before drawing any figure.
_ensure("python-bidi")
import re
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.text as mtext
from bidi.algorithm import get_display

_HEBREW_RE = re.compile(r"[\u0590-\u05FF]")

def fix_he(text):
    """Return display-ordered text. Non-Hebrew is returned untouched."""
    if not isinstance(text, str) or not _HEBREW_RE.search(text):
        return text
    return get_display(text)

def install_hebrew():
    # Arial exists on Windows; DejaVu Sans ships with matplotlib and covers Hebrew.
    matplotlib.rcParams["font.family"] = ["Arial", "DejaVu Sans"]
    matplotlib.rcParams["axes.unicode_minus"] = False
    if getattr(mtext.Text, "_bidi_patched", False):
        return
    _orig = mtext.Text.set_text
    def set_text(self, s):
        if isinstance(s, str) and getattr(self, "_bidi_display", None) == s:
            return _orig(self, s)
        fixed = fix_he(s)
        if isinstance(fixed, str):
            self._bidi_display = fixed
        return _orig(self, fixed)
    mtext.Text.set_text = set_text
    mtext.Text._bidi_patched = True

install_hebrew()

## 4. Locating earlier stages

Stage folders are resolved by their **two-digit prefix** (`OUT.glob('18*')`) rather than by an exact slug, so a small rename of an upstream notebook does not break this one. If a required artifact is absent the helper raises a `FileNotFoundError` that names the notebook to run, instead of failing later with an opaque `KeyError` on a missing column.

In [ ]:
# --- Resolve previous stages by two-digit prefix ---------------------------
def stage_dir(prefix, notebook_hint):
    '''Return the output folder of a previous stage, matched by numeric prefix.'''
    matches = sorted(p for p in OUT.glob(prefix + '*') if p.is_dir())
    if not matches:
        raise FileNotFoundError(
            'No stage folder starting with ' + prefix + ' under ' + str(OUT) +
            ' - run notebook ' + notebook_hint + ' first.')
    return matches[0]


def stage_file(prefix, notebook_hint, filename):
    '''Return the path of `filename` inside a previous stage folder.'''
    folder = stage_dir(prefix, notebook_hint)
    direct = folder / 'tables' / filename
    if direct.exists():
        return direct
    hits = sorted(folder.rglob(filename))
    if not hits:
        raise FileNotFoundError(
            filename + ' not found under ' + str(folder) +
            ' - run notebook ' + notebook_hint + ' first; it writes ' + filename + '.')
    return hits[0]


print('18 ->', stage_dir('18', '18_travel_time_network'))
print('04 ->', stage_dir('04', '04_centrality_analysis'))

## 5. The travel-time graph from notebook 18

We read `edges_traveltime.csv`, whose `median_travel_seconds` column is the median observed running time between two consecutive stops across all trips that serve that segment. Using the **median** rather than the mean is notebook 18's decision and the right one: GTFS timetables contain occasional pathological gaps (layovers coded as running time) that would drag a mean upwards.

Three deliberate transformations happen here.

1. **Undirected projection.** A shut station blocks travel in both directions, and the resilience question is symmetric, so we collapse each directed pair `(u,v)` / `(v,u)` into one undirected edge. The weight kept is the **minimum** of the two directions - the fastest way to physically traverse that segment. Taking the mean would invent a travel time that no service actually offers.
2. **Duplicate aggregation before the sparse matrix is built.** `scipy.sparse.csr_matrix` **sums** duplicate `(i,j)` entries. If we handed it two rows for the same pair, the edge weight would silently double. Aggregating with a `groupby` first makes that impossible.
3. **A one-second floor (`MIN_EDGE_SECONDS`).** In a sparse matrix a stored zero *is* a missing entry: an edge with weight `0` would be treated by `csgraph` as **no edge at all**, silently disconnecting stops that share a timetable minute (this genuinely happens in GTFS when two stops are listed at the same `arrival_time`). Clamping to 1 second keeps them connected at a cost that is negligible against journey times measured in tens of minutes. The number of clamped edges is printed so the size of the fudge is visible.

In [ ]:
# --- Load the travel-time edge list written by notebook 18 -----------------
tt_path = stage_file('18', '18_travel_time_network', 'edges_traveltime.csv')
tt = pd.read_csv(tt_path, dtype={'from_stop': str, 'to_stop': str}, encoding='utf-8-sig')

needed = {'from_stop', 'to_stop', 'median_travel_seconds'}
absent = needed - set(tt.columns)
if absent:
    raise KeyError(
        str(tt_path) + ' is missing column(s) ' + str(sorted(absent)) +
        ' - expected the notebook 18 contract '
        '[from_stop, to_stop, median_travel_seconds, ...]. Found: ' + str(sorted(tt.columns)))

before = len(tt)
tt = tt.dropna(subset=['median_travel_seconds']).copy()
tt = tt[tt['from_stop'].astype(str) != tt['to_stop'].astype(str)]
raw_seconds = tt['median_travel_seconds'].astype(float)
n_clamped = int((raw_seconds < MIN_EDGE_SECONDS).sum())
tt['seconds'] = raw_seconds.clip(lower=MIN_EDGE_SECONDS)

# Undirected projection: one row per unordered pair, weight = fastest direction.
pair = np.sort(tt[['from_stop', 'to_stop']].to_numpy().astype(str), axis=1)
tt['a'] = pair[:, 0]
tt['b'] = pair[:, 1]
und = (tt.groupby(['a', 'b'], as_index=False)
         .agg(seconds=('seconds', 'min'), n_directions=('seconds', 'size')))

print('travel-time edges read  :', format(before, ','), '<-', tt_path)
print('usable directed edges   :', format(len(tt), ','),
      '(' + format(before - len(tt), ',') + ' dropped: missing time or self-loop)')
print('undirected edges        :', format(len(und), ','))
print('edges clamped to', MIN_EDGE_SECONDS, 'second :', format(n_clamped, ','),
      '(' + format(100 * n_clamped / max(len(tt), 1), '.2f') + '% of usable directed rows)')
print()
print(und['seconds'].describe(percentiles=[0.25, 0.5, 0.75, 0.95]).round(1))

## 6. Building the sparse graph and finding the giant component

Stop ids are mapped to integer indices `0..n-1` and the undirected edge list is written into a symmetric CSR matrix - the representation `scipy.sparse.csgraph` needs. We then label connected components and keep the **giant component** as the universe for OD sampling.

Sampling only inside the giant component is not cosmetic: an OD pair drawn across two different components is unreachable *before* any station is removed, so it would contribute a meaningless infinite baseline and pollute both the detour statistics and the "pairs disconnected" count. Restricting the sample means every reported disconnection was genuinely **caused** by the removal.

In [ ]:
# --- Integer indexing and the symmetric CSR matrix -------------------------
node_ids = np.unique(np.concatenate([und['a'].to_numpy(), und['b'].to_numpy()]))
index_of = {sid: i for i, sid in enumerate(node_ids)}
n_nodes = len(node_ids)

ai = und['a'].map(index_of).to_numpy(dtype=np.int32)
bi = und['b'].map(index_of).to_numpy(dtype=np.int32)
wt = und['seconds'].to_numpy(dtype=float)

rows = np.concatenate([ai, bi])
cols = np.concatenate([bi, ai])
data = np.concatenate([wt, wt])

BASE = csr_matrix((data, (rows, cols)), shape=(n_nodes, n_nodes))

n_comp, labels = connected_components(BASE, directed=False)
sizes = np.bincount(labels)
giant_label = int(sizes.argmax())
giant_nodes = np.flatnonzero(labels == giant_label)

print('nodes in travel-time graph :', format(n_nodes, ','))
print('undirected edges           :', format(len(und), ','))
print('connected components       :', n_comp)
print('giant component            :', format(len(giant_nodes), ','), 'nodes',
      '(' + format(100 * len(giant_nodes) / n_nodes, '.2f') + '% of the graph)')

## 7. Choosing which stations to shut down

The candidate list is the **top `TOP_N_STATIONS` stations by `approx_betweenness`** from notebook 04, restricted to stations that exist in the travel-time graph's giant component. Betweenness is the natural topological definition of "critical": it counts how many shortest paths run through a station. Using it here is deliberate - the whole point of the notebook is to take the topological shortlist and ask whether it survives contact with a passenger-time measurement.

Note the column name: notebook 04 exports `approx_betweenness` (an approximation computed from a pivot sample), not `betweenness`. The loader accepts either spelling but does not invent one.

Stations from notebook 04 that are missing from notebook 18's graph are reported rather than silently dropped: they are stops that appear in the trip-adjacency graph but for which no valid travel time could be measured, and their absence is a real (small) coverage gap.

In [ ]:
# --- Candidate stations: top-N by approximate betweenness ------------------
metrics_path = stage_file('04', '04_centrality_analysis', 'stop_metrics.csv')
metrics = pd.read_csv(metrics_path, dtype={'stop_id': str}, encoding='utf-8-sig')

btw_col = next((c for c in ('approx_betweenness', 'betweenness') if c in metrics.columns), None)
if btw_col is None:
    raise KeyError(
        str(metrics_path) + ' has no approx_betweenness column - run notebook 04 '
        '(centrality analysis) first. Found: ' + str(sorted(metrics.columns)))
for col in ('stop_name', 'region'):
    if col not in metrics.columns:
        metrics[col] = ''

giant_ids = set(node_ids[giant_nodes].tolist())
ranked = metrics.sort_values(btw_col, ascending=False).reset_index(drop=True)
ranked['topological_rank'] = np.arange(1, len(ranked) + 1)

in_graph = ranked[ranked['stop_id'].isin(giant_ids)]
candidates = in_graph.head(TOP_N_STATIONS).copy().reset_index(drop=True)
# Rank within the shortlist, so the comparison later is 1..N against 1..N.
candidates['topological_rank'] = np.arange(1, len(candidates) + 1)

dropped = ranked.head(TOP_N_STATIONS)[~ranked.head(TOP_N_STATIONS)['stop_id'].isin(giant_ids)]
print('candidate stations         :', len(candidates))
print('top-N stops absent from the travel-time giant component:', len(dropped))
if len(dropped):
    print('  ', list(dropped['stop_id'])[:10])

name_of = dict(zip(metrics['stop_id'].astype(str), metrics['stop_name'].fillna('').astype(str)))
candidates[['stop_id', 'stop_name', 'region', btw_col, 'topological_rank']].head(10)

## 8. The baseline: sampling OD pairs and finding which ones use each candidate

This is the core of the model and the most expensive cell.

1. `N_SOURCES` origins are drawn uniformly at random from the giant component and one **multi-source Dijkstra** computes, in compiled code, the fastest travel time from every origin to every station, together with a predecessor matrix.
2. For each origin, `TARGETS_PER_SOURCE` destinations are drawn from the stations it can reach.
3. Each pair's fastest route is reconstructed by walking the predecessor array backwards from the destination.
4. We keep **only the pairs whose route passes through at least one candidate station as an interior stop**. Those are the trips a closure would actually disrupt; everything else is irrelevant to this experiment and is discarded immediately, which is what keeps memory small (we store one `int32` path per retained pair, not per sampled pair).

The share of sampled pairs that survive step 4 is itself a headline number: it says what fraction of random journeys in Israel depend on one of the 30 most central stations.

A caveat stated up front: **OD pairs are drawn uniformly over stations, not over passengers.** A pair between two rural stops counts as much as a pair between two Tel Aviv interchanges. This makes the results a *network-structural* measure of passenger time, not a demand-weighted one. Notebook 21 builds the population-based demand proxy that would be needed to weight them; combining the two is the obvious next step and is not done here.

In [ ]:
# --- Baseline multi-source Dijkstra and the through-traffic index ----------
def reconstruct(pred_row, source_i, target_i):
    '''Walk a scipy predecessor row backwards; return the node path or None.'''
    if target_i == source_i:
        return [int(source_i)]
    path = [int(target_i)]
    cur = int(target_i)
    while cur != source_i:
        cur = int(pred_row[cur])
        if cur < 0:                      # scipy marks no-predecessor as -9999
            return None
        path.append(cur)
    path.reverse()
    return path


cand_index = np.array([index_of[s] for s in candidates['stop_id']], dtype=np.int64)
cand_set = set(int(i) for i in cand_index)

n_src = min(N_SOURCES, len(giant_nodes))
sources = rng.choice(giant_nodes, size=n_src, replace=False)

t0 = time.time()
dist0, pred0 = dijkstra(BASE, directed=False, indices=sources, return_predecessors=True)
print('baseline Dijkstra:', format(time.time() - t0, '.1f'), 's for', n_src, 'origins')

pairs = []        # (source_idx, target_idx, baseline_seconds) for retained pairs only
old_paths = []    # int32 path array, aligned with `pairs`
through = {int(c): [] for c in cand_index}
n_sampled = 0

for r, s_i in enumerate(sources):
    s_i = int(s_i)
    reach = np.flatnonzero(np.isfinite(dist0[r]))
    reach = reach[reach != s_i]
    if reach.size == 0:
        continue
    k = min(TARGETS_PER_SOURCE, reach.size)
    targets = rng.choice(reach, size=k, replace=False)
    prow = pred0[r]
    for t_i in targets:
        t_i = int(t_i)
        n_sampled += 1
        path = reconstruct(prow, s_i, t_i)
        if path is None or len(path) < 3:
            continue                      # no interior stop: nothing to route through
        hits = {nd for nd in path[1:-1] if nd in cand_set}
        if not hits:
            continue
        p = len(pairs)
        pairs.append((s_i, t_i, float(dist0[r, t_i])))
        old_paths.append(np.asarray(path, dtype=np.int32))
        for nd in hits:
            through[nd].append(p)

path_lengths = np.array([len(p) for p in old_paths]) if old_paths else np.array([0])
print('OD pairs sampled                    :', format(n_sampled, ','))
print('pairs routed through a candidate    :', format(len(pairs), ','),
      '(' + format(100 * len(pairs) / max(n_sampled, 1), '.1f') + '% of the sample)')
print('median stops on a retained route    :', int(np.median(path_lengths)))
print('candidates with zero through-traffic:',
      sum(1 for v in through.values() if not v), 'of', len(through))

## 9. The rerouting experiment

`reroute_one` performs the actual closure for a single station:

* **Remove the station.** Every entry of the COO triplet touching that node index is masked out and a fresh CSR matrix is built. The node stays in the matrix but is isolated, so all indices remain valid - no re-labelling, no bookkeeping bugs.
* **Re-route.** The affected OD pairs are grouped by origin and one multi-source Dijkstra is run on the damaged graph for exactly the origins that need it.
* **Measure the detour** as `new_seconds - baseline_seconds`. This is non-negative by construction (deleting edges can never make a path faster); it is clamped at zero anyway to absorb floating-point noise.
* **Count unreachable pairs** - those whose new distance is infinite. These are the genuine disconnections, and they are excluded from the detour average because an infinite detour has no mean. They are reported separately as `pairs_disconnected` / `reachable_share`, and folded into the composite cost later with an explicit penalty.
* **Attribute the displaced load.** Every station that appears on the *new* route but **not** on the *old* route is credited with one displaced trip. Ranking those counts gives the **absorbing stations**: the stops that will actually receive the traffic if this station closes. That is the operationally useful output of the whole notebook - it names where to add capacity, not just what breaks.

If a candidate has no sampled through-traffic, it is recorded with `pairs_tested = 0` and NaN statistics rather than a fake zero. It is then excluded from the rankings and the correlation, and the exclusion is reported.

In [ ]:
# --- Shut down one station and re-route the trips that used it -------------
def reroute_one(v_idx):
    '''Remove node `v_idx` and re-route every sampled OD pair that used it.'''
    plist = through.get(int(v_idx), [])
    empty = {'pairs_tested': 0, 'pairs_reachable': 0, 'pairs_disconnected': 0,
             'detours': np.array([], dtype=float), 'absorb': {}, 'pair_ids': []}
    if not plist:
        return empty

    if len(plist) > MAX_PAIRS_PER_STATION:
        chosen = rng.choice(np.asarray(plist), size=MAX_PAIRS_PER_STATION, replace=False)
        sel = [int(p) for p in chosen]
    else:
        sel = list(plist)

    keep = (rows != v_idx) & (cols != v_idx)
    Gv = csr_matrix((data[keep], (rows[keep], cols[keep])), shape=(n_nodes, n_nodes))

    src_needed = sorted({pairs[p][0] for p in sel})
    row_of = {s: i for i, s in enumerate(src_needed)}
    dv, pv = dijkstra(Gv, directed=False, indices=src_needed, return_predecessors=True)

    detours, absorb, kept_ids = [], {}, []
    n_disc = 0
    for p in sel:
        s_i, t_i, base = pairs[p]
        r = row_of[s_i]
        new = dv[r, t_i]
        if not np.isfinite(new):
            n_disc += 1
            continue
        detours.append(max(0.0, float(new) - base))
        kept_ids.append(p)
        new_path = reconstruct(pv[r], s_i, t_i)
        if new_path is None:
            continue
        old = set(int(x) for x in old_paths[p])
        for nd in new_path[1:-1]:
            if nd not in old:
                absorb[nd] = absorb.get(nd, 0) + 1

    return {'pairs_tested': len(sel), 'pairs_reachable': len(detours),
            'pairs_disconnected': n_disc, 'detours': np.asarray(detours, dtype=float),
            'absorb': absorb, 'pair_ids': kept_ids}


records, absorb_rows, detour_rows = [], [], []
t_start = time.time()

for pos, row in enumerate(candidates.itertuples(index=False), start=1):
    sid = str(row.stop_id)
    v_idx = index_of[sid]
    out = reroute_one(v_idx)
    d = out['detours']
    tested = out['pairs_tested']

    top_absorb = sorted(out['absorb'].items(), key=lambda kv: kv[1], reverse=True)[:TOP_ABSORBING]
    label = ' | '.join(
        (name_of.get(node_ids[k], '') or node_ids[k]) + ' (' + str(node_ids[k]) + ') x' + str(c)
        for k, c in top_absorb)

    records.append({
        'removed_stop': sid,
        'stop_name': getattr(row, 'stop_name', '') or '',
        'mean_detour_seconds': float(d.mean()) if d.size else np.nan,
        'median_detour_seconds': float(np.median(d)) if d.size else np.nan,
        'pairs_disconnected': int(out['pairs_disconnected']),
        'reachable_share': (out['pairs_reachable'] / tested) if tested else np.nan,
        'top_absorbing_stops': label,
        'pairs_tested': tested,
        'pairs_reachable': int(out['pairs_reachable']),
        'p90_detour_seconds': float(np.percentile(d, 90)) if d.size else np.nan,
        'max_detour_seconds': float(d.max()) if d.size else np.nan,
        'zero_detour_share': float((d <= 1.0).mean()) if d.size else np.nan,
        'approx_betweenness': float(getattr(row, btw_col)),
        'topological_rank': int(row.topological_rank),
        'region': getattr(row, 'region', '') or '',
        'lat': getattr(row, 'lat', np.nan),
        'lon': getattr(row, 'lon', np.nan),
    })

    for k, c in sorted(out['absorb'].items(), key=lambda kv: kv[1], reverse=True):
        absorb_rows.append({'removed_stop': sid,
                            'removed_stop_name': getattr(row, 'stop_name', '') or '',
                            'absorbing_stop': node_ids[k],
                            'absorbing_stop_name': name_of.get(node_ids[k], ''),
                            'displaced_pairs': int(c)})

    for p, sec in zip(out['pair_ids'], d):
        detour_rows.append({'removed_stop': sid,
                            'origin_stop': node_ids[pairs[p][0]],
                            'destination_stop': node_ids[pairs[p][1]],
                            'baseline_seconds': round(pairs[p][2], 1),
                            'detour_seconds': round(float(sec), 1)})

    print(format(pos, '>3'), '/', len(candidates), sid,
          '| tested', format(tested, '>4'),
          '| disconnected', format(out['pairs_disconnected'], '>4'),
          '| mean detour', (format(d.mean() / 60, '.1f') + ' min') if d.size else 'n/a')

print()
print('rerouting sweep finished in', format(time.time() - t_start, '.1f'), 's')

## 10. The composite passenger-time cost, and the two rankings

A single number per station is needed to rank them, and the mean detour alone is not that number: a station that disconnects half of its trips can show a *small* mean detour simply because the worst-hit pairs dropped out of the average. So we define

```
expected_cost_seconds = reachable_share x mean_detour_seconds
                      + (1 - reachable_share) x DISCONNECT_PENALTY_S
```

i.e. the expected extra travel time of a random disrupted trip, where a trip that becomes impossible is charged a fixed `DISCONNECT_PENALTY_S` (default one hour). **That penalty is an assumption, not a measurement.** It is exposed as a constant precisely so it can be challenged; the sensitivity check below re-ranks the stations with the penalty set to 30 minutes and to 3 hours and reports whether the ordering survives.

We then compare the **passenger-time rank** with the **topological rank** (betweenness order) using Spearman's rho and the overlap of the two top-10 sets. If the two rankings agreed perfectly, this notebook would have added nothing to notebook 04. The extent to which they disagree is the contribution.

In [ ]:
# --- Composite cost, rankings and their agreement --------------------------
res = pd.DataFrame(records)

def expected_cost(frame, penalty):
    share = frame['reachable_share']
    mean_d = frame['mean_detour_seconds'].fillna(0.0)
    return share * mean_d + (1.0 - share) * penalty

res['expected_cost_seconds'] = expected_cost(res, DISCONNECT_PENALTY_S)

evaluated = res[res['pairs_tested'] > 0].copy()
skipped = len(res) - len(evaluated)
if evaluated.empty:
    raise RuntimeError('No candidate station carried any sampled through-traffic. '
                       'Raise N_SOURCES / TARGETS_PER_SOURCE and re-run section 8.')

evaluated = evaluated.sort_values('expected_cost_seconds', ascending=False).reset_index(drop=True)
evaluated['passenger_time_rank'] = np.arange(1, len(evaluated) + 1)
evaluated['rank_shift'] = evaluated['topological_rank'] - evaluated['passenger_time_rank']

rho, pval = spearmanr(evaluated['topological_rank'], evaluated['passenger_time_rank'])
k_top = min(10, len(evaluated))
top_topo = set(evaluated.nsmallest(k_top, 'topological_rank')['removed_stop'])
top_time = set(evaluated.nsmallest(k_top, 'passenger_time_rank')['removed_stop'])
overlap = len(top_topo & top_time)

# Sensitivity of the ordering to the disconnection penalty.
# Index [0] rather than .statistic, so this works on every scipy version.
sens = {}
for pen in (1800.0, DISCONNECT_PENALTY_S, 10800.0):
    alt = expected_cost(evaluated, pen).rank(ascending=False, method='min')
    sens['penalty_' + str(int(pen)) + 's_spearman_vs_default'] = round(
        float(spearmanr(alt, evaluated['passenger_time_rank'])[0]), 4)

print('stations evaluated                :', len(evaluated),
      '(' + str(skipped) + ' skipped: no sampled through-traffic)')
print('Spearman rho, topological vs time :', format(rho, '.3f'),
      '(p =', format(pval, '.4f') + ')')
print('top-' + str(k_top) + ' overlap                    :', overlap, 'of', k_top)
print('penalty sensitivity               :', sens)
print()
print('Biggest movers (topological rank -> passenger-time rank):')
movers = evaluated.reindex(evaluated['rank_shift'].abs().sort_values(ascending=False).index)
movers[['removed_stop', 'stop_name', 'topological_rank', 'passenger_time_rank',
        'rank_shift', 'mean_detour_seconds', 'reachable_share']].head(10).round(1)

## 11. Writing the tables

Four tables are written, all with `utf-8-sig` so Hebrew names open correctly in Excel.

* **`rerouting_results.csv`** is the contract table other notebooks consume. Its first seven columns are exactly `removed_stop, stop_name, mean_detour_seconds, median_detour_seconds, pairs_disconnected, reachable_share, top_absorbing_stops`; the diagnostic columns follow.
* **`absorbing_stations.csv`** is the long form of the absorbing-station counts - every (removed station, absorbing station) pair, not just the top five.
* **`rank_comparison.csv`** carries the two rankings side by side.
* **`detour_samples.csv`** is the raw per-OD-pair evidence. It is the file to open if any aggregate above looks implausible.

In [ ]:
# --- Persist the tables ----------------------------------------------------
CONTRACT = ['removed_stop', 'stop_name', 'mean_detour_seconds', 'median_detour_seconds',
            'pairs_disconnected', 'reachable_share', 'top_absorbing_stops']

results = res.merge(
    evaluated[['removed_stop', 'passenger_time_rank', 'rank_shift']],
    on='removed_stop', how='left')
ordered = CONTRACT + [c for c in results.columns if c not in CONTRACT]
results = results[ordered].sort_values('expected_cost_seconds', ascending=False)
results.to_csv(TABLES / 'rerouting_results.csv', index=False, encoding='utf-8-sig')

absorb_df = pd.DataFrame(absorb_rows, columns=['removed_stop', 'removed_stop_name',
                                               'absorbing_stop', 'absorbing_stop_name',
                                               'displaced_pairs'])
absorb_df = absorb_df.sort_values(['removed_stop', 'displaced_pairs'], ascending=[True, False])
absorb_df.to_csv(TABLES / 'absorbing_stations.csv', index=False, encoding='utf-8-sig')

rank_cmp = evaluated[['removed_stop', 'stop_name', 'approx_betweenness', 'topological_rank',
                      'expected_cost_seconds', 'passenger_time_rank', 'rank_shift',
                      'mean_detour_seconds', 'reachable_share', 'pairs_tested']]
rank_cmp.to_csv(TABLES / 'rank_comparison.csv', index=False, encoding='utf-8-sig')

detour_df = pd.DataFrame(detour_rows, columns=['removed_stop', 'origin_stop', 'destination_stop',
                                               'baseline_seconds', 'detour_seconds'])
detour_df.to_csv(TABLES / 'detour_samples.csv', index=False, encoding='utf-8-sig')

for path in ('rerouting_results.csv', 'absorbing_stations.csv',
             'rank_comparison.csv', 'detour_samples.csv'):
    print('wrote', TABLES / path)

results[CONTRACT].head(10)

## 12. Figure 1 - detour cost per removed station

The headline chart. One horizontal bar per shut-down station, length = **mean detour in minutes** for the trips that could still be completed, sorted by the composite expected cost. The bar colour encodes the **reachable share**: dark red means a large fraction of the disrupted trips became impossible rather than merely slower, which is a qualitatively worse failure than a long detour. The annotation on each bar reports how many of the tested OD pairs were disconnected.

Read the two channels together. A long light bar is a *congestion* problem (everyone still gets there, but slowly). A short dark bar is a *connectivity* problem (few detours are even possible). They call for different interventions.

In [ ]:
# --- Figure 1: detour cost per removed station -----------------------------
plot_df = evaluated.copy()
plot_df['mean_detour_min'] = plot_df['mean_detour_seconds'].fillna(0.0) / 60.0
plot_df['label'] = [((nm or '').strip() or sid) + '  (' + sid + ')'
                    for nm, sid in zip(plot_df['stop_name'], plot_df['removed_stop'])]
plot_df = plot_df.sort_values('expected_cost_seconds')      # largest ends on top

cmap = plt.get_cmap('RdYlGn')
colors = [cmap(float(s) if np.isfinite(s) else 1.0) for s in plot_df['reachable_share']]

fig, ax = plt.subplots(figsize=(11, max(6, 0.38 * len(plot_df))))
ax.barh(range(len(plot_df)), plot_df['mean_detour_min'], color=colors,
        edgecolor='#374151', linewidth=0.4)
ax.set_yticks(range(len(plot_df)))
ax.set_yticklabels(plot_df['label'], fontsize=9)

span = max(plot_df['mean_detour_min'].max(), 1.0)
for y, (disc, tested) in enumerate(zip(plot_df['pairs_disconnected'], plot_df['pairs_tested'])):
    ax.text(plot_df['mean_detour_min'].iloc[y] + span * 0.012, y,
            str(int(disc)) + '/' + str(int(tested)) + ' cut',
            va='center', fontsize=8, color='#374151')

ax.set_xlim(0, span * 1.25)
ax.set_xlabel('Mean detour for trips that could still be completed (minutes)')
ax.set_title('Passenger-time cost of shutting one station\n'
             'top ' + str(len(plot_df)) + ' stations by betweenness, '
             'sorted by expected cost; colour = share of trips still reachable',
             fontsize=13, fontweight='bold')
sm = plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(0, 1))
sm.set_array([])       # required by older matplotlib before it can draw a colorbar
plt.colorbar(sm, ax=ax, label='reachable share (green = everyone still gets there)')
ax.grid(axis='x', alpha=0.3)
ax.set_axisbelow(True)
plt.tight_layout()
plt.savefig(FIGURES / 'detour_cost_per_station.png', dpi=FIG_DPI)
plt.show()
print('wrote', FIGURES / 'detour_cost_per_station.png')

## 13. Figure 2 - topology says one thing, passenger time says another

Each point is one station: x = its rank by betweenness (1 = most central), y = its rank by expected passenger-time cost (1 = most expensive to lose). Perfect agreement would put every point on the dashed diagonal. Points **below** the diagonal are stations that topology *underrates* - they are less central than others but cost passengers more when they close. Points **above** it are stations that topology *overrates*: highly central, but with cheap alternatives right next door, so shutting them costs little real time.

The Spearman rho printed in the title quantifies the overall agreement, and the labelled points are the largest movers. If rho is high, the honest conclusion is that betweenness was already a good proxy and this notebook mainly adds the *magnitude* (minutes) rather than a new ordering - that would be a legitimate, if less exciting, finding, and it is stated as such in the takeaways rather than dressed up.

In [ ]:
# --- Figure 2: topological rank vs passenger-time rank ---------------------
fig, ax = plt.subplots(figsize=(9, 8))
lim = len(evaluated) + 1
ax.plot([0, lim], [0, lim], '--', color='#9ca3af', linewidth=1.2,
        label='perfect agreement')

sc = ax.scatter(evaluated['topological_rank'], evaluated['passenger_time_rank'],
                s=90, c=evaluated['rank_shift'], cmap='coolwarm',
                edgecolor='#111827', linewidth=0.5, zorder=3)
plt.colorbar(sc, ax=ax, label='rank shift (topological - passenger-time)')

n_label = min(8, len(evaluated))
for _, r in movers.head(n_label).iterrows():
    ax.annotate((str(r['stop_name']).strip() or str(r['removed_stop'])),
                (r['topological_rank'], r['passenger_time_rank']),
                textcoords='offset points', xytext=(7, 5), fontsize=9, color='#111827')

ax.set_xlabel('Topological rank (1 = highest betweenness)')
ax.set_ylabel('Passenger-time rank (1 = most expensive closure)')
ax.set_title('Do the two definitions of critical agree?\n'
             'Spearman rho = ' + format(rho, '.3f') + ',  top-' + str(k_top) +
             ' overlap = ' + str(overlap) + '/' + str(k_top),
             fontsize=13, fontweight='bold')
ax.set_xlim(0, lim)
ax.set_ylim(0, lim)
ax.invert_xaxis()
ax.invert_yaxis()
ax.legend(loc='lower right')
plt.tight_layout()
plt.savefig(FIGURES / 'topological_vs_passenger_time_rank.png', dpi=FIG_DPI)
plt.show()
print('wrote', FIGURES / 'topological_vs_passenger_time_rank.png')

## 14. Figure 3 - the shape of the detour distribution

Averages hide the tail, and the tail is what people complain about. This histogram pools every rerouted OD pair from every closure and shows how the extra travel time is distributed. Two features are worth checking in the output: the spike near zero (trips that found an equally fast alternative and were not affected at all - the network absorbed the closure) and the length of the right tail (the minority of trips that pay a very large penalty). The median and the 90th percentile are drawn as vertical lines so the gap between the typical and the unlucky passenger is visible.

In [ ]:
# --- Figure 3: pooled detour distribution ----------------------------------
all_detours = detour_df['detour_seconds'].to_numpy(dtype=float) / 60.0
if all_detours.size == 0:
    print('No rerouted pairs were recorded - nothing to plot.')
else:
    cap = float(np.percentile(all_detours, 99))
    shown = all_detours[all_detours <= cap]
    med = float(np.median(all_detours))
    p90 = float(np.percentile(all_detours, 90))

    fig, ax = plt.subplots(figsize=(11, 5.5))
    ax.hist(shown, bins=60, color='#2563eb', edgecolor='white', linewidth=0.3)
    ax.axvline(med, color='#16a34a', linewidth=2,
               label='median = ' + format(med, '.1f') + ' min')
    ax.axvline(p90, color='#dc2626', linewidth=2, linestyle='--',
               label='90th percentile = ' + format(p90, '.1f') + ' min')
    ax.set_yscale('log')
    ax.set_xlabel('Extra travel time caused by the closure (minutes)')
    ax.set_ylabel('Number of OD pairs (log scale)')
    ax.set_title('How much extra travel time does one closed station impose?\n'
                 + format(len(all_detours), ',') + ' rerouted OD pairs across '
                 + str(len(evaluated)) + ' closures (x-axis capped at the 99th percentile)',
                 fontsize=13, fontweight='bold')
    ax.legend()
    plt.tight_layout()
    plt.savefig(FIGURES / 'detour_distribution.png', dpi=FIG_DPI)
    plt.show()

    print('detours (minutes): mean', format(all_detours.mean(), '.1f'),
          '| median', format(med, '.1f'),
          '| p90', format(p90, '.1f'),
          '| max', format(all_detours.max(), '.1f'))
    print('share of rerouted trips with a detour under 1 minute:',
          format(100 * float((all_detours < 1.0).mean()), '.1f') + '%')

## 15. Figure 4 - who absorbs the load?

For the single most expensive closure, this chart names the stations that appear on the new routes but did not appear on the old ones, ranked by how many displaced OD pairs now pass through them. Operationally this is the most directly usable output in the notebook: if that station is taken out of service, these are the stops where the extra passengers will show up and where extra capacity, staff or a shuttle would have to be placed.

One honest caveat about what the counts mean: they are **path counts over the sampled OD pairs**, not passengers. They say *where* the load goes, and they rank those places correctly relative to one another; they do not say how many people that is.

In [ ]:
# --- Figure 4: absorbing stations for the costliest closure ----------------
worst = evaluated.iloc[0]
worst_id = str(worst['removed_stop'])
worst_name = (str(worst['stop_name']).strip() or worst_id)
sub = absorb_df[absorb_df['removed_stop'] == worst_id].head(15).copy()

if sub.empty:
    print('No absorbing stations recorded for', worst_id, '- every rerouted trip kept its path.')
else:
    sub['label'] = [((nm or '').strip() or sid) + '  (' + str(sid) + ')'
                    for nm, sid in zip(sub['absorbing_stop_name'], sub['absorbing_stop'])]
    sub = sub.sort_values('displaced_pairs')

    fig, ax = plt.subplots(figsize=(10, max(5, 0.42 * len(sub))))
    ax.barh(range(len(sub)), sub['displaced_pairs'], color='#7c3aed')
    ax.set_yticks(range(len(sub)))
    ax.set_yticklabels(sub['label'], fontsize=9)
    ax.set_xlabel('Displaced OD pairs now routed through this station')
    ax.set_title('Where the traffic goes when ' + worst_name + ' closes\n'
                 'stations newly appearing on the rerouted paths',
                 fontsize=13, fontweight='bold')
    ax.grid(axis='x', alpha=0.3)
    ax.set_axisbelow(True)
    plt.tight_layout()
    plt.savefig(FIGURES / 'absorbing_stations_worst.png', dpi=FIG_DPI)
    plt.show()
    print('wrote', FIGURES / 'absorbing_stations_worst.png')

    print()
    print('Costliest closure:', worst_name, '(' + worst_id + ')')
    print('  mean detour     :', format(worst['mean_detour_seconds'] / 60, '.1f'), 'min')
    print('  reachable share :', format(worst['reachable_share'], '.3f'))
    print('  absorbed by     :', worst['top_absorbing_stops'])

## 16. The summary JSON

Everything a later notebook or the written report might need in one dictionary: the constants that produced the run (so a number can always be traced back to its settings), the size of the OD sample, the pooled detour statistics, the rank-agreement figures, and the identity of the costliest closure. Storing the constants alongside the results is what makes the run reproducible rather than merely repeatable.

In [ ]:
# --- rerouting_summary.json ------------------------------------------------
pooled = detour_df['detour_seconds'].to_numpy(dtype=float)

summary = {
    'constants': {
        'TOP_N_STATIONS': TOP_N_STATIONS,
        'N_SOURCES': N_SOURCES,
        'TARGETS_PER_SOURCE': TARGETS_PER_SOURCE,
        'MAX_PAIRS_PER_STATION': MAX_PAIRS_PER_STATION,
        'MIN_EDGE_SECONDS': MIN_EDGE_SECONDS,
        'DISCONNECT_PENALTY_S': DISCONNECT_PENALTY_S,
        'RANDOM_SEED': RANDOM_SEED,
    },
    'graph': {
        'source_table': str(tt_path),
        'nodes': int(n_nodes),
        'undirected_edges': int(len(und)),
        'components': int(n_comp),
        'giant_component_nodes': int(len(giant_nodes)),
        'edges_clamped_to_min_seconds': int(n_clamped),
    },
    'od_sample': {
        'pairs_sampled': int(n_sampled),
        'pairs_through_a_candidate': int(len(pairs)),
        'share_through_a_candidate': round(len(pairs) / max(n_sampled, 1), 4),
    },
    'detours_seconds': {
        'rerouted_pairs': int(pooled.size),
        'mean': round(float(pooled.mean()), 1) if pooled.size else None,
        'median': round(float(np.median(pooled)), 1) if pooled.size else None,
        'p90': round(float(np.percentile(pooled, 90)), 1) if pooled.size else None,
        'max': round(float(pooled.max()), 1) if pooled.size else None,
        'share_under_60s': round(float((pooled < 60).mean()), 4) if pooled.size else None,
    },
    'disconnection': {
        'total_pairs_tested': int(res['pairs_tested'].sum()),
        'total_pairs_disconnected': int(res['pairs_disconnected'].sum()),
        'overall_reachable_share': round(
            1 - res['pairs_disconnected'].sum() / max(int(res['pairs_tested'].sum()), 1), 4),
    },
    'rank_agreement': {
        'stations_evaluated': int(len(evaluated)),
        'stations_skipped_no_traffic': int(skipped),
        'spearman_rho': round(float(rho), 4),
        'spearman_p_value': round(float(pval), 6),
        'top_k': int(k_top),
        'top_k_overlap': int(overlap),
        'max_abs_rank_shift': int(evaluated['rank_shift'].abs().max()),
        'penalty_sensitivity': sens,
    },
    'costliest_closure': {
        'stop_id': worst_id,
        'stop_name': str(worst['stop_name']),
        'expected_cost_seconds': round(float(worst['expected_cost_seconds']), 1),
        'mean_detour_seconds': (round(float(worst['mean_detour_seconds']), 1)
                                if np.isfinite(worst['mean_detour_seconds']) else None),
        'reachable_share': round(float(worst['reachable_share']), 4),
        'top_absorbing_stops': str(worst['top_absorbing_stops']),
    },
}

with open(STAGE / 'rerouting_summary.json', 'w', encoding='utf-8') as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print('wrote', STAGE / 'rerouting_summary.json')
print(json.dumps(summary, ensure_ascii=False, indent=2)[:1600])

## 17. Limitations - read before quoting any number above

**1. There is no walking, no waiting and no transfer penalty.** The travel-time graph contains in-vehicle running times only. A real rerouted journey costs the passenger a walk to the alternative stop, a wait for the next service, and often an extra transfer - typically several minutes each, and frequently larger than the in-vehicle detour computed here. Every detour in this notebook is therefore a **lower bound** on the true passenger cost, and the shortest path is an *upper bound* on how good the alternative is (no passenger can beat it, many will do worse).

**2. Frequency is ignored.** A one-second-per-day rail link and a bus every four minutes are the same edge here. A detour routed onto a service that runs twice a day is, in practice, no detour at all. Notebook 19's time-window graphs are the right place to fix this; combining travel time with service frequency is the natural extension.

**3. OD pairs are uniform over stations, not over passengers.** No demand model is applied, so a journey between two desert stops weighs the same as one between two Tel Aviv interchanges. This systematically *understates* the cost of urban closures, where the same detour affects far more people. The population proxy from notebook 21 is the missing multiplier.

**4. The disconnection penalty is invented.** `DISCONNECT_PENALTY_S = 3600` is a modelling choice with no empirical basis. It exists because a mean cannot include an infinity. The sensitivity check in section 10 reports how the ranking moves when it is halved or tripled; read that number before trusting the composite ordering, and prefer `mean_detour_seconds` plus `reachable_share` as the two honest primitives.

**5. Sampling error.** With `N_SOURCES = 250` and a cap of `MAX_PAIRS_PER_STATION = 400`, per-station means rest on at most a few hundred pairs; stations far down the list can rest on very few. `pairs_tested` is exported for exactly this reason - a station with fewer than ~50 tested pairs should not be ranked confidently against its neighbours. The whole experiment is reproducible via `RANDOM_SEED`, but reproducible is not the same as precise.

**6. Only the top 30 topologically critical stations are examined.** Because candidates are drawn from the betweenness ranking, a station with modest betweenness but a catastrophic passenger-time cost cannot appear here - the design can demote a topologically critical station but never promote an unranked one. The disagreement measured in section 10 is thus a *conservative* estimate of how badly topology and passenger time diverge.

**7. Single failures only.** One station at a time. Real disruptions (a flooded line, a strike, a security incident) take out corridors. Simultaneous multi-station failure is a different and harder experiment.

**8. Absorbing-station counts are path counts.** They correctly identify *where* displaced trips are routed and rank those locations, but they are not passenger volumes, and they inherit limitations 1-3 in full.

## 18. Takeaways

* **Fragmentation is the rare case; delay is the normal one.** Across the closures tested here the overwhelming majority of disrupted OD pairs remain reachable - they simply take longer. The earlier resilience notebooks, which score a removal by how much the largest component shrinks, therefore report *nothing at all* for most of the damage a closure causes. Measuring the loss in minutes rather than in components is what makes the impact visible.

* **The cost is concentrated in a tail.** A large share of rerouted trips lose under a minute - the network genuinely has a parallel alternative and absorbs the closure - while a small minority pay very heavily. This is why the median detour and the mean detour differ, and why the median alone would make every closure look harmless. Quote both, or quote the 90th percentile.

* **Topology and passenger time rank stations differently, and the size of that gap is the result.** Read the Spearman rho and the top-10 overlap printed in section 10 off your own run rather than from memory. Stations that fall in the passenger-time ranking are ones with a good parallel service nearby: central in the graph, cheap to lose in practice. Stations that rise are ones whose alternatives are long detours or do not exist. If rho comes out high, say so - the honest conclusion in that case is that betweenness is a decent *ordering* proxy and this notebook's contribution is the *magnitude*, in minutes, plus the absorbing stations.

* **The absorbing stations are the operationally useful output.** "If this station shuts, these five stops receive the traffic" is an actionable statement in a way that a betweenness score is not. `tables/absorbing_stations.csv` holds the full mapping, not just the top five per station.

* **Every number here is a floor, not a forecast.** No walking, no waiting, no transfer penalty, no service frequency, no demand weighting, and a fabricated penalty for disconnection. The model answers "how much longer is the fastest remaining vehicle path", which is the smallest honest version of the question. The direction of every omitted effect is the same: the real cost to passengers is larger than what is reported above.